# Meeting 11, Act 3: What a Product Costs

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 11.

The three loops of Act 1, the product of NumPy and the GPU of Act 2 all carry out the same multiplications and additions. In this act your group counts these operations, predicts how the time grows with the size of the data, and measures how many operations a second each way of computing reaches. The numbers differ by factors of thousands. The reasons lie in how a computer is built.

## How this lab works

- **This is the notebook of Act 3 of meeting 11.** It starts from its own setup cell. It uses one number from the notebook of Act 2, which Step 17 asks you to copy.
- **Work in the same group.** Each member opens this notebook and runs it. First open the **File** menu and choose **Save a copy in Drive**, so that your changes are kept.
- **This notebook needs no GPU.** Leave its runtime as it is.
- **Run the cells in order**, from the top. To run a cell, click it and press **Shift + Enter**.
- A paragraph that begins with **Your turn** asks you to do something before you go on.
- **In Step 17** your group posts a chart on the discussion board of this meeting. **At the end of the meeting** each member uploads the notebooks to Canvas. The last section of this notebook says how.
- **After the meeting**, **Review Quiz 11** on Canvas asks about this lab. Where the notebook says that the quiz asks about a point, read that part again before you take the quiz.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

In [ ]:
# Setup. Run this cell first.
import time

import matplotlib.pyplot as plt
import numpy as np


def in_words(number):
    """A large number in words: 6.4 million, 1.2 billion, 3.5 trillion."""
    for size, word in ((1e12, "trillion"), (1e9, "billion"), (1e6, "million"), (1e3, "thousand")):
        if number >= size:
            return f"{number / size:.3g} {word}"
    return f"{number:.3g}"


def duration(seconds):
    """A time in the unit that suits it."""
    if seconds < 1:
        return f"{seconds * 1000:.3g} milliseconds"
    if seconds < 120:
        return f"{seconds:.3g} seconds"
    if seconds < 7200:
        return f"{seconds / 60:.3g} minutes"
    if seconds < 172800:
        return f"{seconds / 3600:.3g} hours"
    return f"{seconds / 86400:.3g} days"


print("Setup is done.")

## <font color="#1967d2"><b>Step 16: Count the operations</b></font>

The function `layer_loops` of Act 1 is in the next code cell again. Its innermost line,

`total += W[j, c] * X[i, c]`

carries out one multiplication and one addition. The three loops pass through this line once for every point, every unit and every coordinate: **n × K × m times**.

The cell measures the loops three times, on random numbers: for n = 400 points with m = 30 coordinates and K = 20 units; then with twice as many points; then with n, m and K all doubled.

**Your turn.** Before you run the cell, answer in the next cell.

1. When n is doubled, by what factor does the time grow?
2. When n, m and K are all doubled, by what factor does the time grow, compared with the first measurement?

**Our predictions.** Double-click this cell and replace this line with your two factors.

In [ ]:
def layer_loops(X, W, b):
    """The scores of the K units of a layer on the n points of X, one multiplication at a time."""
    n, m = X.shape
    K = W.shape[0]
    S = np.empty((n, K))
    for i in range(n):                   # every point
        for j in range(K):               # every unit
            total = b[j]
            for c in range(m):           # every coordinate
                total += W[j, c] * X[i, c]
            S[i, j] = total
    return S


rng = np.random.default_rng(0)           # a source of random numbers
cases = [(400, 30, 20), (800, 30, 20), (800, 60, 40)]
loop_times = []
print(f"{'n':>6s}{'m':>5s}{'K':>5s}{'n x K x m':>13s}{'seconds':>10s}{'ratio to the first':>21s}{'nanoseconds per pass':>23s}")
for n, m, K in cases:
    X, W, b = rng.random((n, m)), rng.random((K, m)), rng.random(K)
    start = time.perf_counter()
    layer_loops(X, W, b)
    loop_times.append(time.perf_counter() - start)
    print(f"{n:6d}{m:5d}{K:5d}{n * K * m:13,d}{loop_times[-1]:10.3f}{loop_times[-1] / loop_times[0]:21.1f}"
          f"{loop_times[-1] / (n * K * m) * 1e9:23.0f}")

**How to read the result.** The time grows as the number of passes does: by a factor near 2 when n is doubled, and by a factor near 8 when all three numbers are doubled. Measured times vary by some per cent from run to run, so the factors are not exact. The last column, the time of one pass, is nearly the same in the three rows. So

time ≈ (the time of one pass) × n × K × m.

Computer science writes this as **O(nKm)**, read "of the order of n K m". The notation keeps how the number of operations grows with the sizes, and it drops the constant in front, the time of one pass. For a layer with n = m = K the count is n³: when the size doubles, the time is multiplied by 8. That is the factor you found in Act 2 between the sizes 2,048 and 4,096.

The page **[What matrix operations cost](https://ikoutis.github.io/cs301-labs/unit-2/pages/matrix-costs.html)** lists this count for the operations of this unit.

## <font color="#1967d2"><b>Step 17: The same operations, in other seconds</b></font>

The count says how the time **grows**. It does not say how long one operation takes. The next cells measure that for five ways of computing, as the number of **arithmetic operations per second**, where each multiplication and each addition counts as one operation:

1. **Python, three loops:** the first measurement of Step 16;
2. **NumPy, `a + b` on long arrays:** two arrays of about 16 million numbers each are added, four times;
3. **NumPy, `a + b` on short arrays:** two arrays of about 16 thousand numbers each are added, 4,096 times, which is the same number of additions as in the line above;
4. **NumPy, the product `X @ W.T`** with n = m = K = 2,048;
5. **the GPU, the same product** at size 4,096, from your measurement in Act 2.

**Your turn.** In the notebook of Act 2, find in the table of Step 15 the seconds that the GPU took at size 4,096. Type that number in the next cell in place of `None`, and run the cell. If your group has no measurement of a GPU, leave `None`.

In [ ]:
GPU_SECONDS_AT_4096 = None       # replace None with the seconds of the GPU at size 4096, from Step 15 of Act 2

assert GPU_SECONDS_AT_4096 is None or 0 < GPU_SECONDS_AT_4096 < 60, "Type the seconds as a number, for example 0.0312."

In [ ]:
rng = np.random.default_rng(1)
rates = {}                               # for each way of computing: arithmetic operations per second

# 1. Python, three loops: one multiplication and one addition for each pass
n, m, K = cases[0]
rates["Python: three loops"] = 2 * n * K * m / loop_times[0]

# 2. NumPy, a + b on long arrays: one addition for each pair of numbers
long = 2 ** 24                           # 16,777,216 numbers in each array
a = rng.random(long, dtype=np.float32)
b = rng.random(long, dtype=np.float32)
c = np.empty(long, dtype=np.float32)
np.add(a, b, out=c)                      # c = a + b, with the result written into the array c; not measured
start = time.perf_counter()
for _ in range(4):
    np.add(a, b, out=c)
rates["NumPy: a + b, long arrays"] = 4 * long / (time.perf_counter() - start)

# 3. NumPy, a + b on short arrays: the same number of additions
short = 2 ** 14                          # 16,384 numbers in each array
a, b, c = a[:short].copy(), b[:short].copy(), c[:short].copy()
calls = 4 * long // short
np.add(a, b, out=c)
start = time.perf_counter()
for _ in range(calls):
    np.add(a, b, out=c)
rates["NumPy: a + b, short arrays"] = calls * short / (time.perf_counter() - start)

# 4. NumPy, the product of a layer: one multiplication and one addition for each of size x size x size passes
size = 2048
X = rng.random((size, size), dtype=np.float32)
W = rng.random((size, size), dtype=np.float32)
X @ W.T                                  # not measured
start = time.perf_counter()
for _ in range(2):
    X @ W.T
rates["NumPy: X @ W.T"] = 2 * 2 * size ** 3 / (time.perf_counter() - start)

# 5. the GPU, from Act 2
if GPU_SECONDS_AT_4096:
    rates["GPU: X @ W.T"] = 2 * 4096 ** 3 / GPU_SECONDS_AT_4096

slowest = min(rates.values())
print(f"{'way of computing':30s}{'operations per second':>24s}{'ratio to the slowest':>23s}")
for name, rate in rates.items():
    print(f"{name:30s}{in_words(rate):>24s}{rate / slowest:23,.0f}")

In [ ]:
names = list(rates)
values = list(rates.values())

fig, ax = plt.subplots(figsize=(9, 0.62 * len(names) + 1.5))
ax.hlines(names, min(values) / 3, values, color="#c3c2b7", linewidth=1)      # a thin line from the edge to each mark
ax.plot(values, names, "o", color="#2a78d6", markersize=9)
for name, value in zip(names, values):
    ax.annotate(in_words(value), (value, name), xytext=(10, 0), textcoords="offset points", va="center")
ax.set_xscale("log")
ax.set_xlim(min(values) / 3, max(values) * 40)
ticks = [10.0 ** e for e in range(6, 14) if min(values) / 3 <= 10.0 ** e <= max(values) * 40]
ax.set_xticks(ticks)
ax.set_xticklabels([in_words(tick) for tick in ticks])
ax.invert_yaxis()                        # the first way of computing at the top
ax.set_xlabel("arithmetic operations per second (each mark on the axis is 10 times the one before)")
ax.set_title("How many additions and multiplications a second?")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print("Operations per second. " + "; ".join(f"{name}: {in_words(value)}" for name, value in rates.items()) + ".")

**How to read the chart.** Every mark counts the same thing: additions and multiplications of two numbers in one second. The axis is stretched, as in Act 2: each mark on it is ten times the one before. The table above the chart holds the same numbers.

- **Python against NumPy.** The loops run in the interpreter, which does a great deal of work around each operation. Meeting 10 measured this.
- **Long arrays against short arrays.** These two rows run the same additions, by the same code. Only the place of the numbers differs. A processor has a small and fast memory of its own, called the **cache**. The short arrays fit into it and stay there. The long arrays do not fit: each number has to be fetched from the main memory of the computer, is used in **one** addition, and the result is sent back. The processor spends most of its time waiting for numbers. Such a computation is **memory-bound**: the memory sets its speed, not the arithmetic.
- **The product.** In `X @ W.T` a number is not used once: each entry of `X` is multiplied with a weight of every unit. NumPy's product is programmed to work on blocks of the two matrices that fit into the cache, and to use each number many times while it is there. The processor spends its time on arithmetic. Such a computation is **compute-bound**. Here every core of the processor helps, and so do its instructions that handle several numbers at once.
- **The GPU.** Thousands of small cores work on the same product. This pays for compute-bound work on large arrays, and, as you measured in Act 2, not for small ones.

The notation O( ) is a first estimate of the cost of a computation, and the right one to begin with. The marks of the chart show what it leaves out: how the program is run, where the numbers are, and which processor does the work. The page **[CPU and GPU](https://ikoutis.github.io/cs301-labs/unit-2/pages/cpu-and-gpu.html)** shows what is inside the two processors. Meeting 12 looks at the memory of a computer.

**Your turn: post now.** Your group posts on the discussion board of this meeting, as a reply to its post of Act 1. The post contains:

- the line that the cell printed under the chart;
- the chart. To save it, right-click it and choose **Save image as**;
- under the chart, **one sentence that says what it shows**, so that a classmate who cannot see the chart still gets the result;
- one sentence on the two rows that your group finds most surprising.

## <font color="#1967d2"><b>Step 18: Predict the time of a layer you cannot try</b></font>

The count and the rates together predict the time of a computation before it is run:

time ≈ (number of operations) ÷ (operations per second).

Take a layer of K = 1,000 units, applied to n = 60,000 small grey images of 28 by 28 pixels, so that m = 784.

**Your turn.** Before you run the next cell, compute in the cell after this one:

1. the number of operations, 2 × n × K × m;
2. the time that the three loops in Python would need, from your rate in Step 17. Give it in hours.

**Our answers.** Double-click this cell and replace this line with the number of operations and the time in hours.

In [ ]:
n, m, K = 60_000, 784, 1_000
operations = 2 * n * K * m
print(f"n = {n:,}, m = {m}, K = {K:,}: {operations:,} operations ({in_words(operations)})")
print(f"{'way of computing':30s}{'predicted time':>22s}")
for name, rate in rates.items():
    if "a + b" not in name:              # the two rows of additions are not ways of computing a product
        print(f"{name:30s}{duration(operations / rate):>22s}")

**How to read the result.** The three times belong to the same computation, with the same O( ). **Review Quiz 11** asks you to make such a prediction for another layer, and to say which of the measurements of this act is memory-bound and which is compute-bound.

## At the end of the meeting: hand in

**Your group has posted three times** on the discussion board of this meeting: at Step 7 in Act 1, at Step 15 in Act 2, and at Step 17 in Act 3. If one of the three posts is missing, add it now.

**Read the posts of two other groups.** Their machines are not yours: compare their operations per second with yours.

**Each member hands in the three notebooks.** In the notebooks of Acts 1, 2 and 3, open the **File** menu, choose **Download**, then **Download .ipynb**. Upload the three files to this meeting's class-work assignment on Canvas.

**After the meeting**, take **Review Quiz 11** on Canvas. Some of its questions ask for an answer of a few sentences. Your notebooks, with your group's measurements, prompts and pictures in them, are the reference for them.